# Clearvale: download stim.csv files and one subject's fMRIPrep outputs from Flywheel

Step 1 of running the Kim et al. (2020) template RSA on Clearvale. This notebook only **lists and downloads**; nothing on Flywheel is changed.

1. **Log in** with your Flywheel API key (typed into a hidden prompt; never stored in this notebook).
2. **Find** the Clearvale project and pick one subject.
3. **Inventory**: every `stim*.csv` file attached to that subject's containers, and the fMRIPrep analyses on its sessions, with sizes.
4. **Select** the fMRIPrep files the RSA needs (preprocessed BOLD, brain masks, confounds, anatomical segmentations) from inside the output zip, without downloading the whole zip.
5. **Download** -- only after you have looked at the tables and set `DOWNLOAD = True`.

Run the cells top to bottom. Use the `~/.venvs/rsaworkflow` kernel (or any Python 3 kernel).

## 0. Install the Flywheel SDK (once per kernel)

In [ ]:
import flywheel

## 1. Settings

Change these, then re-run from here. `PROJECT_PATTERN` and `SUBJECT` can stay as they are for the first pass: the next cells list what they match.

In [ ]:
import os
import re
from pathlib import Path

# -- what to look for on Flywheel --
PROJECT_PATTERN = r"CV"        # regex on project labels (case-insensitive)
GROUP = None                          # e.g. "banich" to restrict to one group; None = any group you can see
SUBJECT = "090"                        # subject label, e.g. "1001"; None = list the subjects and stop
STIM_PATTERN = r"stim.*\.csv$"        # regex on file names (case-insensitive)
FMRIPREP_GEAR_PATTERN = r"bids-hcp"  # regex on the analysis gear name / label (Clearvale preprocessing ran as bids-hcp)
OUTPUT_ZIP_PATTERN = r"_hcp\.zip$"     # the output zip with the preprocessed data (NOT pipeline_logs.zip)

KEEP_ALL = True                       # whole subject: download the zip once (29.4 GB) and unpack everything (32.8 GB)
DELETE_ZIP_AFTER_EXTRACT = True       # remove the downloaded zip once it is unpacked

# -- which preprocessed files to pull out of the output zip when KEEP_ALL is False (BIDS names; check section 7a for what the zip holds) --
SPACES = ["MNI152NLin6Asym", "T1w"]   # output spaces to keep (Clearvale's MVPA used MNI152NLin6Asym; Kim 2020 used native space)
KEEP_PATTERNS = [
    rf"func/.*_space-({'|'.join(SPACES)})(_res-[^_]+)?_desc-preproc_bold\.(nii\.gz|json)$",
    rf"func/.*_space-({'|'.join(SPACES)})(_res-[^_]+)?_desc-brain_mask\.nii\.gz$",
    rf"func/.*_space-({'|'.join(SPACES)})(_res-[^_]+)?_boldref\.nii\.gz$",
    r"func/.*_desc-confounds_timeseries\.(tsv|json)$",
    rf"anat/.*(_space-({'|'.join(SPACES)}))?(_res-[^_]+)?_(desc-aparcaseg_|desc-aseg_)?dseg\.nii\.gz$",
    rf"anat/.*(_space-({'|'.join(SPACES)}))?(_res-[^_]+)?_desc-(preproc_T1w|brain_mask)\.nii\.gz$",
    r"dataset_description\.json$",
    r"sub-[^/]+\.html$",
]

# -- where to put it: this folder (rsaworkflow/data/cv-bids -> ~/data/rsaworkflow/cv-bids,
#    git-ignored and outside OneDrive) --
# the Clearvale BIDS folder: $CV_BIDS, else <repo>/data/cv-bids (run this notebook from examples/clearvale/notebooks)
OUT = Path(os.environ.get("CV_BIDS", Path.cwd().resolve().parents[2] / "data" / "cv-bids"))
DOWNLOAD = True                       # section 8 downloads
ALLOW_FULL_ZIP = False                # only if the SDK can't extract zip members: download whole zips, then extract

OUT.mkdir(parents=True, exist_ok=True)
print("downloads go to", OUT)

## 2. Log in

In [ ]:
import getpass
import flywheel

fw = flywheel.Client()

me = fw.get_current_user()
print("logged in as", me.email)

## 3. Find the project

In [ ]:
import pandas as pd

projects = [p for p in fw.projects() if re.search(PROJECT_PATTERN, p.label, re.I)
            and (GROUP is None or p.group == GROUP)]
display(pd.DataFrame([{"group": p.group, "project": p.label, "id": p.id} for p in projects]))
if len(projects) != 1:
    raise SystemExit(f"{len(projects)} projects match; narrow PROJECT_PATTERN / GROUP and re-run")
project = projects[0]
print(f"using {project.group}/{project.label}")

## 4. Pick a subject

In [ ]:
subjects = project.subjects()
print(f"{len(subjects)} subjects:", ", ".join(sorted(s.label for s in subjects)[:60]), "..." if len(subjects) > 60 else "")
if SUBJECT is None:
    raise SystemExit("set SUBJECT in the settings cell to one of the labels above, then re-run from there")
matches = [s for s in subjects if s.label == str(SUBJECT)]
if not matches:
    raise SystemExit(f"no subject labelled {SUBJECT!r}")
subject = matches[0].reload()
sessions = [s.reload() for s in subject.sessions()]
print(f"subject {subject.label}: sessions", [s.label for s in sessions])

## 5. Inventory: stim files and fMRIPrep analyses

Walks the project, the subject, its sessions and their acquisitions (files) and the subject's and sessions' analyses (outputs). Nothing is downloaded.

In [ ]:
def human(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"


def analyses_of(container):
    try:
        return list(container.analyses or [])
    except Exception:
        return []


stim_rows, analysis_rows = [], []
levels = [("project", project.reload(), ""), ("subject", subject, "")]
for ses in sessions:
    levels.append(("session", ses, ses.label))
    for acq in ses.acquisitions():
        levels.append(("acquisition", acq.reload(), f"{ses.label}/{acq.label}"))

for level, cont, where in levels:
    for f in cont.files or []:
        if re.search(STIM_PATTERN, f.name, re.I):
            stim_rows.append({"level": level, "where": where, "file": f.name, "size": f.size,
                              "_container": cont})
    if level in ("subject", "session"):
        for an in analyses_of(cont):
            gear = (an.gear_info or {}) if hasattr(an, "gear_info") else {}
            name = gear.get("name", "") if isinstance(gear, dict) else getattr(gear, "name", "")
            version = gear.get("version", "") if isinstance(gear, dict) else getattr(gear, "version", "")
            if re.search(FMRIPREP_GEAR_PATTERN, f"{name} {an.label}", re.I):
                for f in an.files or []:
                    analysis_rows.append({"level": level, "where": where, "analysis": an.label, "gear": f"{name} {version}",
                                          "created": an.created, "file": f.name, "size": f.size, "_analysis": an})

stim = pd.DataFrame(stim_rows)
fmriprep = pd.DataFrame(analysis_rows)
print(f"stim files: {len(stim)}, total {human(stim['size'].sum() if len(stim) else 0)}")
display(stim.drop(columns="_container", errors="ignore"))
print(f"fMRIPrep analysis outputs: {len(fmriprep)}")
display(fmriprep.drop(columns="_analysis", errors="ignore").assign(size=lambda d: d["size"].map(human)) if len(fmriprep) else fmriprep)

If no stim files show up, they may sit somewhere else (another project, a different file name): widen `STIM_PATTERN`, or set `PROJECT_PATTERN` to that project.

## 6. Choose the fMRIPrep output zip(s)

Takes the newest fMRIPrep analysis per container that has a `.zip` output. Override `CHOSEN` by hand if you want a different run.

In [ ]:
zips = fmriprep[fmriprep["file"].str.contains(OUTPUT_ZIP_PATTERN, regex=True)] if len(fmriprep) else fmriprep
if zips.empty:
    raise SystemExit("no output zip matches OUTPUT_ZIP_PATTERN; check the file names in the table above")
# newest analysis per container; among its matching zips, the largest (all outputs of one analysis share a timestamp)
CHOSEN = zips.sort_values(["created", "size"]).groupby("where", as_index=False).tail(1)
display(CHOSEN.drop(columns="_analysis").assign(size=lambda d: d["size"].map(human)))
assert not CHOSEN["file"].str.contains("log", case=False).any(), "picked a logs zip; fix OUTPUT_ZIP_PATTERN"

## 7. Pick the files the RSA needs from inside each zip

In [ ]:
keep_rx = [re.compile(p) for p in KEEP_PATTERNS]


def bids_relpath(member):
    """Path inside the zip -> path under derivatives/, keeping the pipeline folder that holds the
    subject (e.g. bids-hcp/sub-090/..., HCPPipe/sub-090/..., fmriprep/sub-090/...). bids-hcp's
    BIDS-named files are symlinks into the HCPPipe tree, so the two trees must stay side by side."""
    parts = member.split("/")
    for i, part in enumerate(parts):
        if part.startswith("sub-") or part == "dataset_description.json":
            return "/".join(parts[max(i - 1, 0):])
    return "/".join(parts[-2:])


member_rows = []
zip_member_ok = True
for _, row in CHOSEN.iterrows():
    an = row["_analysis"]
    try:
        info = an.get_file_zip_info(row["file"])
    except Exception as err:  # older SDKs / storage without zip listing
        print(f"(!) cannot list members of {row['file']}: {err}")
        zip_member_ok = False
        continue
    for m in info.members:
        if m.path.endswith("/"):
            continue
        if KEEP_ALL or any(rx.search(m.path) for rx in keep_rx):
            member_rows.append({"zip": row["file"], "member": m.path, "dest": bids_relpath(m.path), "size": m.size,
                                "_analysis": an})

members = pd.DataFrame(member_rows)
if len(members):
    members = members.drop_duplicates("dest")
    print(f"{len(members)} files selected, total {human(members['size'].sum())}")
    display(members.drop(columns="_analysis").assign(size=lambda d: d["size"].map(human)))
if not zip_member_ok:
    print(f"whole-zip fallback would download {human(CHOSEN['size'].sum())} (set ALLOW_FULL_ZIP = True)")

### 7a. What is in the zip, by file type

Every file in the chosen zip(s), with subject / session / task / run / dir values replaced by `*`, so each kind of file is one row (count and total size). Use it to adjust `SPACES` / `KEEP_PATTERNS` (e.g. whether bids-hcp wrote confounds, masks or native-space BOLD), then re-run the settings cell and section 7. Rows marked `kept` are what section 8 will download.

In [ ]:
kind_rows = []
for _, row in CHOSEN.iterrows():
    try:
        info = row["_analysis"].get_file_zip_info(row["file"])
    except Exception as err:
        print(f"(!) cannot list members of {row['file']}: {err}")
        continue
    for m in info.members:
        if m.path.endswith("/"):
            continue
        rel = bids_relpath(m.path)
        kind = re.sub(r"(sub|ses|task|run|dir|acq|echo)-[^_/.]+", r"\1-*", rel)
        kind_rows.append({"zip": row["file"], "path": m.path, "kind": kind, "size": m.size,
                          "kept": KEEP_ALL or any(rx.search(m.path) for rx in keep_rx)})
all_members = pd.DataFrame(kind_rows)
all_members.to_csv(OUT / "zip_members.csv", index=False)  # full listing, for choosing KEEP_PATTERNS
kinds = (all_members.groupby("kind").agg(n=("size", "size"), size=("size", "sum"), kept=("kept", "max"))
         .sort_values("size", ascending=False).reset_index())
print(f"{int(kinds['n'].sum())} files in {len(kinds)} kinds, {human(kinds['size'].sum())} in total; "
      f"kept: {int(kinds.loc[kinds['kept'], 'n'].sum())} files, {human(kinds.loc[kinds['kept'], 'size'].sum())}")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 140):
    display(kinds.assign(size=kinds["size"].map(human)))
print("full member list written to", OUT / "zip_members.csv")

## 8. Download

Review the tables above first. Then set `DOWNLOAD = True` in the settings cell, re-run the settings cell, and run this one. With `KEEP_ALL = True` the output zip is downloaded once and unpacked here (one 29 GB transfer instead of ~13,500 single-file requests); the zip is then deleted if `DELETE_ZIP_AFTER_EXTRACT`. Symlinks in the zip stay symlinks. Files that already exist with the same size are skipped, so re-running is safe (re-running after the zip was deleted downloads it again).

Layout under `OUT`:
- `sourcedata/stim/<level path>/<file>` -- the stim.csv files
- `derivatives/<pipeline folder>/sub-.../...` -- the gear's own trees, side by side (for bids-hcp: `derivatives/bids-hcp/` with the BIDS names, symlinked into `derivatives/HCPPipe/`)
- `download_manifest.csv` -- what came from where

In [ ]:
import os
import stat
import zipfile

if not DOWNLOAD:
    raise SystemExit("DOWNLOAD is False: review the tables, then set DOWNLOAD = True and re-run settings + this cell")

manifest = []


def fetch(dest: Path, size, getter):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and size and dest.stat().st_size == size:
        return "skipped (exists)"
    getter(str(dest))
    return "downloaded"


for _, row in stim.iterrows():
    dest = OUT / "sourcedata" / "stim" / row["level"] / (row["where"] or subject.label) / row["file"]
    status = fetch(dest, row["size"], lambda p, r=row: r["_container"].download_file(r["file"], p))
    manifest.append({"kind": "stim", "source": f"{row['level']}:{row['where']}/{row['file']}", "dest": str(dest), "status": status})
    print(status, dest)

deriv = (OUT / "derivatives").resolve()  # members keep their pipeline folder (bids_relpath)


def unpack(zpath: Path, wanted):
    """Extract the members `wanted(name)` selects, at their BIDS-relative paths under `deriv`.
    Members stored as symlinks (bids-hcp's BIDS-named files point into MNINonLinear/...) are
    recreated as symlinks, not as tiny files holding the target path."""
    n_files = n_links = 0
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            name = info.filename
            if name.endswith("/") or not wanted(name):
                continue
            dest = Path(os.path.normpath(deriv / bids_relpath(name)))
            if deriv not in dest.parents:
                print("(!) skipped member outside the target folder:", name)
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            if stat.S_ISLNK(info.external_attr >> 16):
                target = zf.read(info).decode()
                if os.path.lexists(dest):
                    os.remove(dest)
                os.symlink(target, dest)
                n_links += 1
                status = f"symlink -> {target}"
            else:
                if dest.exists() and dest.stat().st_size == info.file_size:
                    status = "skipped (exists)"
                else:
                    with zf.open(info) as src, open(dest, "wb") as out:
                        while chunk := src.read(16 * 1024 * 1024):
                            out.write(chunk)
                    status = "extracted"
                n_files += 1
            manifest.append({"kind": "derivative", "source": f"{zpath.name}:{name}", "dest": str(dest), "status": status})
    print(f"{zpath.name}: {n_files} files, {n_links} symlinks -> {deriv}")


if KEEP_ALL or (not len(members) and ALLOW_FULL_ZIP):
    for _, row in CHOSEN.iterrows():
        zpath = OUT / "zips" / row["file"]
        print(f"downloading {row['file']} ({human(row['size'])}) -- this takes a while")
        print(fetch(zpath, row["size"], lambda p, r=row: r["_analysis"].download_file(r["file"], p)), zpath)
        unpack(zpath, (lambda name: True) if KEEP_ALL else (lambda name: any(rx.search(name) for rx in keep_rx)))
        if DELETE_ZIP_AFTER_EXTRACT:
            zpath.unlink()
            print("deleted", zpath)
elif len(members):
    for _, row in members.iterrows():
        dest = deriv / row["dest"]
        status = fetch(dest, row["size"],
                       lambda p, r=row: r["_analysis"].download_file_zip_member(r["zip"], r["member"], p))
        manifest.append({"kind": "derivative", "source": f"{row['zip']}:{row['member']}", "dest": str(dest), "status": status})
        print(status, dest)

pd.DataFrame(manifest).to_csv(OUT / "download_manifest.csv", index=False)
print(f"done: {len(manifest)} entries; manifest at {OUT / 'download_manifest.csv'}")

## 9. Check what arrived

In [ ]:
for path in sorted(OUT.rglob("*")):
    if path.is_file() and "zips" not in path.parts and path.suffix != ".ipynb":
        print(f"{human(path.stat().st_size):>10}  {path.relative_to(OUT)}")

## 10. Flag the files the RSA will use

Classifies every downloaded file by the rules below (first match wins) and writes `file_inventory.csv` next to this notebook: path, size, role, and `use` (True for the files the Kim 2020 RSA reads). Edit `USE_RULES` / `USE_ROLES` and re-run; nothing is moved or deleted.

In [ ]:
USE_SPACE = "MNI152NLin6Asym"   # space the RSA runs in (Clearvale's MVPA used MNI152NLin6Asym)
USE_RULES = [  # (role, regex on the path relative to OUT); first match wins
    ("stim",            r"^sourcedata/stim/.*\.csv$"),
    ("bold",            rf"/func/.*_space-{USE_SPACE}(_res-[^_]+)?_desc-preproc_bold\.nii(\.gz)?$"),
    ("bold_sidecar",    rf"/func/.*_space-{USE_SPACE}(_res-[^_]+)?_desc-preproc_bold\.json$"),
    ("bold_mask",       rf"/func/.*_space-{USE_SPACE}(_res-[^_]+)?_desc-brain_mask\.nii(\.gz)?$"),
    ("boldref",         rf"/func/.*_space-{USE_SPACE}(_res-[^_]+)?_boldref\.nii(\.gz)?$"),
    ("confounds",       r"/func/.*(confounds|regressors|motion|movement|Movement_Regressors)[^/]*\.(tsv|txt|csv|json)$"),
    ("events",          r"_events\.tsv$"),
    ("anat_seg",        r"/anat/.*(dseg|aparc|aseg|wmparc)[^/]*\.nii(\.gz)?$"),
    ("anat_t1w",        rf"/anat/.*(_space-{USE_SPACE})?(_res-[^_]+)?_desc-preproc_T1w\.nii(\.gz)?$"),
    ("anat_mask",       r"/anat/.*_desc-brain_mask\.nii(\.gz)?$"),
    ("bold_other_space", r"/func/.*_desc-preproc_bold\.nii(\.gz)?$"),
    ("report",          r"\.html$"),
]
USE_ROLES = {"stim", "bold", "bold_sidecar", "bold_mask", "boldref", "confounds", "events", "anat_seg"}

rows = []
for path in sorted(OUT.rglob("*")):
    if not path.is_file() or "zips" in path.parts or path.suffix == ".ipynb" or path.name == "file_inventory.csv":
        continue
    rel = "/" + str(path.relative_to(OUT))
    role = next((r for r, rx in USE_RULES if re.search(rx, rel[1:] if rx.startswith("^") else rel)), "other")
    rows.append({"path": rel[1:], "size": path.stat().st_size, "role": role, "use": role in USE_ROLES})
inventory = pd.DataFrame(rows)
inventory.to_csv(OUT / "file_inventory.csv", index=False)
print(f"{len(inventory)} files, {human(inventory['size'].sum())}; "
      f"flagged for use: {int(inventory['use'].sum())} files, {human(inventory.loc[inventory['use'], 'size'].sum())}")
display(inventory.groupby(["use", "role"]).agg(n=("path", "size"), size=("size", "sum"))
        .assign(size=lambda d: d["size"].map(human)))
print("inventory written to", OUT / "file_inventory.csv")